```bash
# Install SAM2
pip install git+https://github.com/facebookresearch/sam2.git
```

In [ ]:
import os
import random

import torch
import numpy as np
from torch.utils.data import Dataset
from PIL import Image

class MassHumanBurns(Dataset):
    _views_per_burn = 4
    def __init__(self, root_dir, return_full_masks=True, return_burn_masks=True):
        self.root_dir = root_dir
        self._pairs_per_burn = self._views_per_burn ** 2
        self.labels = torch.tensor(np.load(os.path.join(root_dir, "labels.npy")), dtype=torch.float32)
        self.return_full_masks = return_full_masks
        self.return_burn_masks = return_burn_masks
        self._len = self.labels.numel() * self._pairs_per_burn
        self._burn_levels = self.labels.shape[1]
        self._pairs_per_human = self._burn_levels * self._pairs_per_burn

    def __len__(self):
        return self._len

    def __getitem__(self, idx):
        human_idx = idx // self._pairs_per_human
        burn_idx = idx % self._pairs_per_human // self._pairs_per_burn
        front_view_idx = idx % self._pairs_per_human % self._pairs_per_burn // self._views_per_burn
        back_view_idx = idx % self._pairs_per_human % self._pairs_per_burn % self._views_per_burn
        res = {"label": self.labels[human_idx, burn_idx]}
        if self.return_full_masks:
            res["full_masks"] = (
                Image.open(os.path.join(self.root_dir, "full_masks", "front", f"{human_idx}_{burn_idx}_{front_view_idx}.png")), 
                Image.open(os.path.join(self.root_dir, "full_masks", "back", f"{human_idx}_{burn_idx}_{back_view_idx}.png"))
            )
        if self.return_burn_masks:
            res["burn_masks"] = (
                Image.open(os.path.join(self.root_dir, "burn_masks", "front", f"{human_idx}_{burn_idx}_{front_view_idx}.png")), 
                Image.open(os.path.join(self.root_dir, "burn_masks", "back", f"{human_idx}_{burn_idx}_{back_view_idx}.png"))
            )
        return res

ds = MassHumanBurns("mass_human_burns")


In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

sam2_model = build_sam2("configs/sam2.1/sam2.1_hiera_t.yaml", "sam2.1_hiera_tiny.pt")
sam2predictor = SAM2ImagePredictor(sam2_model)

In [ ]:
def pad_to_square(image, size=512):
    w, h = image.size
    l = (size - w) // 2
    t = (size - h) // 2
    result = Image.new(image.mode, (size, size), 0)
    result.paste(image, (l, t))
    return result

In [ ]:
from skimage import measure

def get_box(mask):
    y_min, x_min = np.where(mask)[0].min(), np.where(mask)[1].min()
    y_max, x_max = np.where(mask)[0].max(), np.where(mask)[1].max()
    return x_min, y_min, x_max, y_max

def convert_to_SAM_precessed_mask(mask):
    sam2predictor.set_image(np.array(mask.convert("RGB")))
    labels, num = measure.label(np.array(mask), background=0, return_num=True)
    masks = [torch.from_numpy(labels == i + 1) for i in range(num)]
    new_masks = []
    for m in masks:
        x_min, y_min, x_max, y_max = get_box(m)
        try:
            new_mask = sam2predictor.predict(box=torch.tensor([x_min, y_min, x_max, y_max]), multimask_output=False)[0][0]
        except:
            continue
        new_masks.append(new_mask)
    if len(new_masks) == 0:
        return np.zeros_like(mask)
    return np.stack(new_masks).max(0).astype(np.bool_)

def crop_by_mask(image, mask):
    x_min, y_min, x_max, y_max = get_box(mask)
    return image.crop((x_min, y_min, x_max, y_max))

In [ ]:
import math
from tqdm import tqdm

os.makedirs(os.path.join("sam_precessed_MHB", "burn_masks", "front"), exist_ok=True)
os.makedirs(os.path.join("sam_precessed_MHB", "burn_masks", "back"), exist_ok=True)
os.makedirs(os.path.join("sam_precessed_MHB", "full_masks", "front"), exist_ok=True)
os.makedirs(os.path.join("sam_precessed_MHB", "full_masks", "back"), exist_ok=True)

for i in tqdm(range(len(ds))):
    human_idx = i // ds._pairs_per_human
    burn_idx = i % ds._pairs_per_human // ds._pairs_per_burn
    view_idx = i % ds._pairs_per_human % ds._pairs_per_burn
    if view_idx not in [0, 5, 10, 15]:
        continue
    front_view_idx = view_idx // ds._views_per_burn
    back_view_idx = view_idx % ds._views_per_burn
    size = random.randint(512, int(512*math.sqrt(2)))

    front_burn_mask = ds[i]["burn_masks"][0]
    back_burn_mask = ds[i]["burn_masks"][1]
    front_full_mask = ds[i]["full_masks"][0]
    back_full_mask = ds[i]["full_masks"][1]

    front_burn_mask_padded = pad_to_square(front_burn_mask, size)
    back_burn_mask_padded = pad_to_square(back_burn_mask, size)
    front_full_mask_padded = pad_to_square(front_full_mask, size)
    back_full_mask_padded = pad_to_square(back_full_mask, size)

    sam_precessed_front_burn_mask = Image.fromarray(convert_to_SAM_precessed_mask(front_burn_mask_padded))
    sam_precessed_back_burn_mask = Image.fromarray(convert_to_SAM_precessed_mask(back_burn_mask_padded))
    sam_precessed_front_full_mask = Image.fromarray(convert_to_SAM_precessed_mask(front_full_mask_padded))
    sam_precessed_back_full_mask = Image.fromarray(convert_to_SAM_precessed_mask(back_full_mask_padded))

    sam_precessed_front_burn_mask_cropped = crop_by_mask(sam_precessed_front_burn_mask, sam_precessed_front_full_mask)
    sam_precessed_back_burn_mask_cropped = crop_by_mask(sam_precessed_back_burn_mask, sam_precessed_back_full_mask)
    sam_precessed_front_full_mask_cropped = crop_by_mask(sam_precessed_front_full_mask, sam_precessed_front_full_mask)
    sam_precessed_back_full_mask_cropped = crop_by_mask(sam_precessed_back_full_mask, sam_precessed_back_full_mask)

    sam_precessed_front_burn_mask_cropped.save(os.path.join("sam_precessed_MHB", "burn_masks", "front", f"{human_idx}_{burn_idx}_{front_view_idx}.png"))
    sam_precessed_back_burn_mask_cropped.save(os.path.join("sam_precessed_MHB", "burn_masks", "back", f"{human_idx}_{burn_idx}_{back_view_idx}.png"))
    sam_precessed_front_full_mask_cropped.save(os.path.join("sam_precessed_MHB", "full_masks", "front", f"{human_idx}_{burn_idx}_{front_view_idx}.png"))
    sam_precessed_back_full_mask_cropped.save(os.path.join("sam_precessed_MHB", "full_masks", "back", f"{human_idx}_{burn_idx}_{back_view_idx}.png"))